In [3]:
import sys
import os
import pandas as pd
import networkx as nx
import numpy as np

# Make sure src is importable if needed later
sys.path.append(os.path.abspath(".."))

# Load parquet directly for now (no src abstraction yet)
df = pd.read_parquet("../data/problem_a.parquet")

G = nx.Graph()
for _, row in df.iterrows():
    G.add_edge(row["node_1"], row["node_2"], weight=float(row["weight"]))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 21
Edges: 28


In [4]:
#Build a coherent chunk (BFS)

from collections import deque
import networkx as nx

def bfs_chunk(G, seed, K=10):
    seen = {seed}
    q = deque([seed])
    while q and len(seen) < K:
        u = q.popleft()
        for v in G.neighbors(u):
            if v not in seen:
                seen.add(v); q.append(v)
                if len(seen) >= K: break
    return seen

seed = sorted(G.nodes())[0]
nodes_chunk = bfs_chunk(G, seed, K=10)
Gsub = G.subgraph(nodes_chunk).copy()
print("chunk:", Gsub.number_of_nodes(), "nodes,", Gsub.number_of_edges(), "edges")

chunk: 10 nodes, 12 edges


In [5]:
#Random score for Max Cut

import random

def score_cut(Gx, a):
    return sum(float(d.get("weight",1.0)) for u,v,d in Gx.edges(data=True) if a[u]!=a[v])

nodes = sorted(Gsub.nodes())
best = -1
for _ in range(200):
    a = {n: random.randint(0,1) for n in nodes}
    best = max(best, score_cut(Gsub, a))
print("random best(200):", best)


random best(200): 1524.1356216943982


In [ ]:

#simple greedy search

def greedy_maxcut(Gx):
    nodes = sorted(Gx.nodes())
    # start random
    import random
    assign = {n: random.randint(0,1) for n in nodes}

    improved = True
    while improved:
        improved = False
        for n in nodes:
            current = score_cut(Gx, assign)
            assign[n] ^= 1  # flip
            new = score_cut(Gx, assign)
            if new < current:
                assign[n] ^= 1  # revert
            else:
                improved = True
    return assign, score_cut(Gx, assign)

g_assign, g_score = greedy_maxcut(Gsub)
print("Greedy score:", g_score)

Greedy score: 1559.5438419276293


In [7]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import Aer

def qaoa_p1_maxcut(Gx, gamma, beta):
    nodes = sorted(Gx.nodes())
    idx = {n:i for i,n in enumerate(nodes)}
    n = len(nodes)
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for u,v,d in Gx.edges(data=True):
        w = float(d.get("weight",1.0))
        i,j = idx[u], idx[v]
        qc.cx(i,j); qc.rz(2*gamma*w, j); qc.cx(i,j)
    for q in range(n):
        qc.rx(2*beta, q)
    qc.measure(range(n), range(n))
    return qc, nodes

backend = Aer.get_backend("aer_simulator")
qc, nodes = qaoa_p1_maxcut(Gsub, gamma=0.6, beta=0.8)
res = backend.run(qc, shots=500).result()
counts = res.get_counts()
list(counts.items())[:5]

[('1110011011', 1),
 ('1011011011', 1),
 ('1000110100', 1),
 ('0111010001', 2),
 ('0001111110', 1)]

In [8]:
best_q = -1
for bitstring, c in counts.items():
    bits = list(map(int, bitstring[::-1]))   # qiskit order
    a = {nodes[i]: bits[i] for i in range(len(nodes))}
    best_q = max(best_q, score_cut(Gsub, a))
print("best qaoa sample:", best_q)

best qaoa sample: 1608.553597369426


In [9]:
gammas = np.linspace(0.1, 1.5, 6)
betas = np.linspace(0.1, 1.5, 6)

best_overall = -1

for g in gammas:
    for b in betas:
        qc, nodes = qaoa_p1_maxcut(Gsub, g, b)
        res = backend.run(qc, shots=300).result()
        counts = res.get_counts()
        for bitstring in counts:
            bits = list(map(int, bitstring[::-1]))
            a = {nodes[i]: bits[i] for i in range(len(nodes))}
            val = score_cut(Gsub, a)
            best_overall = max(best_overall, val)

print("Best over grid:", best_overall)

Best over grid: 1613.7724798401232


In [10]:
print("random best(200):", 1524.1356216943982)
print("greedy:", 1559.5438419276293)
print("qaoa best sample (single):", 1608.553597369426)
print("qaoa best over grid:", 1613.7724798401232)

random best(200): 1524.1356216943982
greedy: 1559.5438419276293
qaoa best sample (single): 1608.553597369426
qaoa best over grid: 1613.7724798401232
